# Finishing Position Model — EDA & Evaluation
Lightweight check on the training data and model performance.

In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

df = pd.read_csv('data_finishing.csv')
model_bundle = joblib.load('../models/finishing.pkl')

model      = model_bundle['model']
le_driver  = model_bundle['le_driver']
le_circuit = model_bundle['le_circuit']

print('=== DATA OVERVIEW ===')
print(f'Rows         : {len(df)}')
print(f'Seasons      : {sorted(df["season"].unique())}')
print(f'Circuits     : {df["circuit"].nunique()} unique')
print(f'Drivers      : {df["driver"].nunique()} unique')
print(f'Nulls        : {df.isnull().sum().sum()}')

In [ ]:
# Re-encode and split the same way as training
df = df.dropna()
df['driver_enc']  = le_driver.transform(df['driver'])
df['circuit_enc'] = le_circuit.transform(df['circuit'])

X = df[['driver_enc', 'circuit_enc', 'season', 'grid_position']]
y = df['finishing_position']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

y_pred = model.predict(X_test)
diff   = np.abs(y_pred - y_test)

print('=== MODEL EVALUATION ===')
print(f'Test rows         : {len(X_test)}')
print(f'Exact accuracy    : {accuracy_score(y_test, y_pred):.2%}')
print(f'Within 1 position : {(diff <= 1).mean():.2%}')
print(f'Within 3 positions: {(diff <= 3).mean():.2%}')
print(f'Within 5 positions: {(diff <= 5).mean():.2%}')
print(f'Mean abs error    : {diff.mean():.2f} positions')

In [ ]:
print('=== FEATURE IMPORTANCES ===')
for feat, imp in sorted(zip(X.columns, model.feature_importances_), key=lambda x: -x[1]):
    bar = '█' * int(imp * 40)
    print(f'{feat:<15} {imp:.4f}  {bar}')

In [ ]:
print('=== KNOWN DRIVERS & CIRCUITS ===')
print(f'Drivers  ({len(le_driver.classes_)}):', list(le_driver.classes_))
print()
print(f'Circuits ({len(le_circuit.classes_)}):', list(le_circuit.classes_))